In [1]:
import anndata as ad 
import pandas as pd
import numpy as np


gw15 = ad.read_h5ad("/Users/sydneycole/neuro/neuro/data/gw15.h5ad")
gw20 = ad.read_h5ad("/Users/sydneycole/neuro/neuro/data/gw20.h5ad")
gw22 = ad.read_h5ad("/Users/sydneycole/neuro/neuro/data/gw22.h5ad")
gw34 = ad.read_h5ad("/Users/sydneycole/neuro/neuro/data/gw34.h5ad")
ba17 = ad.read_h5ad("/Users/sydneycole/neuro/neuro/data/gw34_umb5900_ba17.h5ad", backed='r')
norm_exp = ad.read_h5ad("/Users/sydneycole/neuro/neuro/data/norm_exp.h5ad")
main_merfish = ad.read_h5ad("/Users/sydneycole/neuro/neuro/data/merscope_integrated_855.h5ad", backed="r")


print(f"\n== gw15 ==\nShape: {gw15.shape}\nObs list: {gw15.obs.columns.to_list()}\nAnnotation example: {gw15.obs.iloc[0].to_dict()}")
print(f"\n== gw20 ==\nShape: {gw20.shape}\nObs list: {gw20.obs.columns.to_list()}\nAnnotation example: {gw20.obs.iloc[0].to_dict()}")
print(f"\n== gw22 ==\nShape: {gw22.shape}\nObs list: {gw22.obs.columns.to_list()}\nAnnotation example: {gw22.obs.iloc[0].to_dict()}")
print(f"\n== gw34 ==\nShape: {gw34.shape}\nObs list: {gw34.obs.columns.to_list()}\nAnnotation example: {gw34.obs.iloc[0].to_dict()}")
print(ba17.obs.columns.tolist())
print(ba17.obs[['gw', 'sample', 'region', 'area']].head() if 'area' in ba17.obs.columns else ba17.obs.head())

print(f"\n== Merscope integrated (master MERFISH file) == \nShape: {main_merfish.shape}")
print("Obs columns:", main_merfish.obs.columns.tolist())
print("Obsm keys:", list(main_merfish.obsm.keys()))
print("Layers:", list(main_merfish.layers.keys()))
print("Cell counts by section/donor/timepoint:")
for col in ['donor_id', 'sample', 'gw', 'ges`tational_week', 'cortical_area']:
    if col in main_merfish.obs.columns:
        print(f"  {col}:", main_merfish.obs[col].value_counts().head(10))

print(f"\n== Normalized expression == \nShape: {norm_exp.shape}")
print("Has spatial coords?", 'spatial' in norm_exp.obsm)
print("Number of genes:", norm_exp.n_vars)  # ~300 = MERFISH, >15000 = snRNA-seq

/opt/anaconda3/envs/neurospatial/lib/python3.11/site-packages/anndata/_core/anndata.py:1882: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/opt/anaconda3/envs/neurospatial/lib/python3.11/site-packages/anndata/_core/anndata.py:1882: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")



== gw15 ==
Shape: (5139902, 300)
Obs list: ['gw', 'sample', 'region', 'H1_annotation', 'H2_annotation', 'H3_annotation']
Annotation example: {'gw': '15', 'sample': 'UMB1117', 'region': 'F1a', 'H1_annotation': 'EC', 'H2_annotation': 'EC', 'H3_annotation': nan}

== gw20 ==
Shape: (8044446, 300)
Obs list: ['gw', 'sample', 'region', 'H1_annotation', 'H2_annotation', 'H3_annotation', 'v1_v2_dist', 'cortical_depth']
Annotation example: {'gw': '20', 'sample': 'FB080', 'region': 'F1', 'H1_annotation': 'EN-IT', 'H2_annotation': 'EN-IT-L3/4', 'H3_annotation': 'EN-IT-L3/4-c5', 'v1_v2_dist': nan, 'cortical_depth': nan}

== gw22 ==
Shape: (2612118, 300)
Obs list: ['gw', 'sample', 'region', 'H1_annotation', 'H2_annotation', 'H3_annotation']
Annotation example: {'gw': '22', 'sample': 'FB123', 'region': 'F1', 'H1_annotation': 'IN', 'H2_annotation': 'IN-VZ', 'H3_annotation': nan}

== gw34 ==
Shape: (1173061, 300)
Obs list: ['gw', 'sample', 'region', 'H1_annotation', 'H2_annotation', 'H3_annotation']
A

/opt/anaconda3/envs/neurospatial/lib/python3.11/site-packages/anndata/_core/anndata.py:1882: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


## Terminology

# Dataset Terminology & Reference

This notebook works with the Qian/Walsh et al. 2025 MERFISH atlas of the developing human cortex (Nature, https://doi.org/10.1038/s41586-025-09010-1) - data deposited at Zenodo (10.5281/zenodo.14422018). The atlas spans gestational weeks 15, 20, 22, and 34 across eight cortical areas, with paired snRNA-seq and Visium data.
---

## Dataset Structure

### File naming conventions
- **`gw##`** — gestational week (GW15, GW18, GW20, GW21, GW22, GW34)
- **`FB###`** — fetal brain donor ID (e.g., FB080, FB121, FB123)
- **`UMB####`** — donor ID from the University of Maryland Brain & Tissue Bank (e.g., UMB1117, UMB5900)
- **`BA##`** — Brodmann area, the standard cortical regional nomenclature
- **Section codes (F1, F2, O1, P1, T1, F1a, etc.)** — anatomical position of a tissue section within a sample; F = frontal, O = occipital, P = parietal, T = temporal; the number indicates sequential sections



## Cortical Anatomy

### Brodmann areas in this dataset
- **BA4** — primary motor cortex
- **BA9** — dorsolateral prefrontal cortex
- **BA17** — primary visual cortex (V1)
- **BA18** — secondary visual cortex (V2)
- **BA22** — superior temporal gyrus / auditory association
- **BA40** — supramarginal gyrus (parietal association)
- **BA123** — primary somatosensory cortex (combined BAs 1, 2, 3)

The paper's central finding involves a discrete molecular boundary between BA17 (V1) and BA18 (V2) as early as GW20.

### Mature cortical layers (for reference — not directly in this dataset)
The developing brain doesn't have these layers yet, but understanding their adult counterparts helps interpret the layer-fated developing neurons (e.g., EN-IT-L4 = an L4-fated neuron):
- **L1** — derived from the marginal zone; sparse cell bodies, dense neuropil
- **L2/3** — superficial layers; intratelencephalic (IT) projection neurons connecting cortical regions
- **L4** — granular layer; receives thalamocortical input, especially prominent in sensory areas
- **L5** — deep projection layer; contains pyramidal tract (PT) and large IT neurons
- **L6** — deepest layer; contains corticothalamic (CT) neurons and L6 IT neurons

---


## Developmental zones / layers (in this dataset)

Listed from innermost (ventricular surface, where neurons are born) to outermost (pial surface, future cortical surface). The annotation uses developmental zone names rather than mature anatomical terms; e.g., there is no "L1" in the developing cortex.

### Germinal zones (where new cells are born)
- **vz** (337K) — ventricular zone; innermost germinal layer adjacent to the ventricle; contains apical radial glia (vRG, the founder progenitor pool)
- **isvz** (362K) — inner subventricular zone
- **osvz** (1.4M, largest layer) — outer subventricular zone; contains outer radial glia (oRG); massively expanded in humans and central to cortical surface expansion

### Migratory and developing zones
- **iz** (976K) — intermediate zone; migration corridor for newborn neurons traveling to the cortical plate; becomes white matter in the mature brain
- **sp** (454K) — subplate; transient layer beneath the cortical plate; important for thalamocortical circuit development; largely disappears by adulthood

### Cortical plate (CP — where neurons settle to form the future six-layer cortex)
Layers form inside-out: deep layers first, superficial layers last. Newborn neurons migrate past older ones to reach more superficial positions.
- **L6** (765K), **L5** (402K), **L4** (401K), **L3** (364K), **L2** (167K)

### Outermost
- **mz** (79K) — marginal zone; will become L1 of the mature cortex; contains Cajal-Retzius cells

---


## Cell Types

### Annotation hierarchy
Cell type labels are provided at three levels of increasing specificity:
- **H1** — major cell class (e.g., `EN-IT`, `IN`, `RG`)
- **H2** — subclass with developmental or spatial context (e.g., `EN-IT-L3/4`, `IN-VZ`, `oRG1`)
- **H3** — finest subtype, often a specific cluster (e.g., `EN-IT-L3/4-c5`); NaN where finer subdivision isn't assigned

### Reading cell type labels
Labels often encode location, fate, or lineage:
- **EN-IT-L4** = excitatory neuron, intratelencephalic projection, layer-4-fated
- **EN-Mig-oSVZ-1** = migrating excitatory neuron currently in the outer SVZ
- **IN-MGE** = inhibitory neuron from the medial ganglionic eminence
- **IN-VZ/GE** = inhibitory neuron transiting the VZ or still in the ganglionic eminences
- **vRG / oRG / tRG** = ventricular / outer / truncated radial glia subtypes

---

### H1 classes (the 8 major cell types in this dataset)

**Excitatory neuron lineage:**
- **EN-IT** (3.6M) — intratelencephalic excitatory neurons; glutamatergic projection neurons connecting cortical regions (including callosal projections). Found across L2–L6. The largest excitatory class.
- **EN-Mig** (3.4M) — migrating excitatory neurons; newborn projection neurons traversing the IZ toward the cortical plate. Distinct from settled EN classes because their transcriptomic state during migration is different.
- **EN-ET** (2.2M) — extratelencephalic excitatory neurons; project to subcortical targets (brainstem, spinal cord). Likely includes corticothalamic neurons within EN-ET-L6-early or EN-ET-L5/6, since no separate EN-CT class exists; the dataset emphasizes the subplate component of EN-ET, with 3 of 5 subtypes capturing distinct subplate populations

**Neural progenitors:**
- **IPC** (2.1M) — intermediate progenitor cells; transit-amplifying progenitors in the SVZ that divide symmetrically to amplify neuronal output.
- **RG** (1.8M) — radial glia; major neural progenitor population. H2 subtypes resolve into vRG, oRG, and tRG (see below).

**Inhibitory lineage:**
- **IN** (2.0M) — inhibitory neurons; GABAergic, originate from the ganglionic eminences (MGE, CGE, LGE) and migrate tangentially into the cortex. H2 subtypes resolve by origin (MGE/CGE) and migratory state (VZ/GE).

**Non-neuronal:**
- **Glia** (436K) — non-RG glial cells; in this dataset includes OPCs and astrocyte precursors. The H2 labels (Astro-1, Astro-late1, OPC, etc.) resolve these further.
- **EC** (368K) — endothelial cells; blood vessel cells.

Note: Microglia (MG), pericytes (Peric/Mural), and vascular leptomeningeal cells (VLMC) do not appear as distinct H1 classes — either lumped under Glia/EC or sparsely sampled given the 300-gene MERFISH panel doesn't focus on them.

---


### Key H2 subtypes (verified for this dataset)

**Excitatory neuron lineage:**

*EN-IT* — layer-fated cortical projection neurons:
- EN-IT-L2/3, EN-IT-L3/4, EN-IT-L4, EN-IT-L4/5, EN-IT-L6

*EN-Mig* — migrating excitatory neurons, by current location along the migration trajectory:
- EN-oSVZ-1, EN-oSVZ-2 (just leaving birthplace; note typo in data: "En-oSVZ-2" with lowercase n)
- EN-IZ-1, EN-IZ-2 (passing through the intermediate zone)
- EN-L2 (nearly at destination in cortical plate; layer-2-fated)

*EN-ET* — deep-layer and subplate extratelencephalic neurons:
- EN-ET-L5/6 (classical deep-layer projection neurons)
- EN-ET-L6-early (early-born L6 neurons; may include corticothalamic)
- EN-ET-SP, EN-ET-SP-P, EN-ET-SP-early (three subplate populations — transient, mid-gestation peak, important for thalamocortical circuit establishment)

**Neural progenitors:**

*RG* — radial glia, with the gliogenic transition captured:
- RG1 (generic / early vRG)
- vRG-late (late-stage ventricular RG)
- oRG1 (outer radial glia; human-expanded population)
- tRG (truncated RG)
- Astro-late1 (RG transitioning to astrocyte fate — captures the neurogenic-to-gliogenic switch)

*IPC* — intermediate progenitors, by zone and molecular state:
- IPC-VZ/SVZ (boundary), IPC-iSVZ (inner SVZ), IPC-oSVZ (outer SVZ)
- IPC-SVZ-1, IPC-SVZ-2 (two molecular subtypes within the SVZ)

**Inhibitory lineage (IN):**
- INP-VZ/GE (interneuron progenitors at birthplace)
- IN-VZ/GE (early migratory, in VZ or ganglionic eminences)
- IN-MGE (medial ganglionic eminence-derived; future parvalbumin+ / somatostatin+)
- IN-CGE (caudal ganglionic eminence-derived; future VIP+ / reelin+)
- IN-SST (somatostatin+, already differentiating)

**Non-neuronal:**
- *Glia* (H1): Astro-1 (early astrocyte precursors), OPC (oligodendrocyte precursors). No microglia resolved.
- *EC* (H1): single class, no subtypes.

Note: Microglia, pericytes, and vascular leptomeningeal cells are not resolved in this dataset (likely below detection given the 300-gene MERFISH panel).

---


## Spatial Metadata Columns (in master integrated file)

- **`gw`** — gestational week
- **`sample`** — donor ID
- **`region`** — anatomical section identifier within a donor
- **`area`** — Brodmann area assignment
- **`layer`** — cortical layer assignment (zone or layer)
- **`relative_height`** — normalized position along the cortical column (0 = ventricular surface, 1 = pial surface, typically)
- **`cortical_depth`** — absolute distance measure from a reference surface
- **`v1_v2_dist`** — signed distance from the V1/V2 boundary (GW20 only); negative on one side, positive on the other; central to the paper's V1/V2 sharp-boundary finding

`obsm['spatial']` holds the 2D xy coordinates per cell (per-section coordinate origins).
---


## Methods and Technologies

### Spatial transcriptomics platforms
- **MERFISH (Multiplexed Error-Robust Fluorescence In Situ Hybridization)** — imaging-based spatial transcriptomics; detects individual mRNA molecules with sub-micron resolution; limited to a pre-designed gene panel (300 genes in this dataset). Commercial platform: Vizgen MERSCOPE.
- **Visium** — 10x Genomics spot-based spatial transcriptomics; ~55µm spots capture whole-transcriptome (~20,000 genes) but at multi-cell resolution
- **Stereo-seq** — BGI's spatial transcriptomics platform; sub-cellular resolution, whole-transcriptome (alternative to MERFISH/Visium, used in other developmental brain atlases)

### Single-cell methods used as references
- **scRNA-seq** — single-cell RNA sequencing on dissociated cells
- **snRNA-seq** — single-nucleus RNA sequencing (used in the Qian/Walsh paper because fetal brain tissue is hard to dissociate intact); profiles nuclear transcripts

### Analytical methods (project pipeline)
- **scVI / scANVI** — variational autoencoder methods for scRNA-seq integration and label transfer
- **UCE (Universal Cell Embedding)** — foundation model providing pretrained cell embeddings; alternatives: scGPT, Geneformer
- **Banksy** — spatial clustering using cells' own and neighborhood expression; standard spatial domain method
- **STAGATE / GraphST** — graph neural network methods for spatial domain identification
- **cell2location / RCTD / Tangram** — cell type deconvolution methods (relevant for Visium, not needed for MERFISH which has single-cell resolution)
- **CellChat / CellPhoneDB** — non-spatial cell-cell communication inference using ligand-receptor databases
- **COMMOT** — spatially-aware cell-cell communication using optimal transport
- **ENVI** — environmental variational inference; imputes whole-transcriptome expression onto MERFISH cells using paired snRNA-seq
---


## Important neurobiological notes

- **Inside-out neurogenesis**: cortical layers form in reverse order. Deep layers (L6, L5) are born first from RG; superficial layers (L4, L3, L2) are born later. New neurons migrate past older ones to reach more superficial positions.
- **Interneuron origin**: cortical interneurons (IN) are NOT born in the cortex. They originate in the ganglionic eminences (GE) of the ventral telencephalon and undertake long tangential migrations to reach the cortex. In this dataset, IN cells in the VZ are migratory, not progenitor-derived.
- **Human-specific features**: the oSVZ is massively expanded relative to mouse; outer radial glia (oRG) are abundant in humans and underlie cortical surface area expansion.
- **Time window**: GW15 = mid-neurogenesis; GW20–22 = peak neurogenesis with layer specification underway; GW34 = late gestation, near-complete neurogenesis with maturing circuitry.
---


## References for terminology

- Molnár et al. (2019) *Science* — review of human cortical development principles
- Lui, Hansen & Kriegstein (2011) *Cell* — outer radial glia and cortical expansion
- Bystron, Blakemore & Rakic (2008) *Nature Reviews Neuroscience* — developmental cortical zone definitions
- Qian/Walsh et al. (2025) *Nature* — this dataset, primary source for atlas-specific terminology

In [8]:
# Verification: actual annotation values in this dataset
for col in ['H1_annotation', 'area', 'layer']:
    if col in main_merfish.obs.columns:
        print(f"\n{col} ({main_merfish.obs[col].nunique()} unique):")
        print(main_merfish.obs[col].value_counts().to_string())


H1_annotation (8 unique):
H1_annotation
EN-IT     3626848
EN-Mig    3373685
EN-ET     2167483
IPC       2137317
IN        2021276
RG        1796244
Glia       436449
EC         368068

area (16 unique):
area
A-PFC     1595266
A-Par      907043
A-Temp     551819
B-V2       473030
PFC        412017
A-V1       338030
B-Cing     281704
A-Occi     251701
B-Par      190344
A-M1       180109
A-S1       173450
A-V2       149903
B-V1       113719
C-V2        54202
A-Cing      50133
B-M1        32696

layer (11 unique):
layer
osvz    1392192
iz       976347
l6       765064
sp       453677
l5       402340
l4       401291
l3       363905
isvz     362464
vz       337433
l2       167356
mz        78895


In [ ]:
for h1 in main_merfish.obs['H1_annotation'].unique():
    h2_list = main_merfish.obs[main_merfish.obs['H1_annotation']==h1]['H2_annotation'].unique()
    h2_list = [h for h in h2_list if pd.notna(h)]
    print(f"\n{h1} ({len(h2_list)} H2 subtypes):")
    for h2 in sorted(h2_list):
        print(f"  - {h2}")

In [ ]:
import pandas as pd

print(pd.crosstab(main_merfish.obs['sample'], main_merfish.obs['area']))

In [ ]:
for h1 in ['EN-IT', 'EN-Mig', 'RG', 'IN']:
    print(f"\n=== H2 subtypes of {h1} ===")
    h2_counts = main_merfish.obs[main_merfish.obs['H1_annotation']==h1]['H2_annotation'].value_counts()
    print(h2_counts.head(15))

In [ ]:
print(f"area NaN count: {ba17.obs['area'].isna().sum()}")
print(f"area total: {len(ba17.obs)}")
print(f"area unique non-NaN values: {ba17.obs['area'].dropna().unique()}")

print(f"region unique values: {ba17.obs['region'].unique()}")
print(f"region counts: {ba17.obs['region'].value_counts()}")

# Are non-NaN area cells concentrated in specific layers?
print("Layer distribution of cells WITH area assignment:")
print(ba17.obs[ba17.obs['area'].notna()]['layer'].value_counts())

print("\nLayer distribution of cells WITHOUT area assignment:")
print(ba17.obs[ba17.obs['area'].isna()]['layer'].value_counts())

# Are non-NaN cells concentrated in specific H1 types?
print("\nH1 distribution of cells WITH area assignment:")
print(ba17.obs[ba17.obs['area'].notna()]['H1_annotation'].value_counts())

print("\nH1 distribution of cells WITHOUT area assignment:")
print(ba17.obs[ba17.obs['area'].isna()]['H1_annotation'].value_counts())

print("All unique layer values in BA17 file:")
print(ba17.obs['layer'].value_counts(dropna=False))

In [ ]:
# In notebook 01
from neurospatial.harmonization import harmonize_master_celltypes
from neurospatial.data_loading import get_project_root, load_main_merfish

PROJECT_ROOT = get_project_root()
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# Reload master in backed mode, no harmonization yet
main_merfish = load_main_merfish(PROJECT_ROOT, harmonized=False)

# Apply harmonization (obs_names_make_unique was called in load_main_merfish)
main_merfish = harmonize_master_celltypes(main_merfish)

# Save the harmonized column
output_path = DATA_PROCESSED / 'main_merfish_obs_harmonized.csv'
main_merfish.obs['harmonized'] = main_merfish.obs['harmonized'].fillna('Not_in_taxonomy')
main_merfish.obs[['harmonized']].to_csv(output_path)
print(f"Saved to: {output_path}")
print(f"\nHarmonized distribution:")
print(main_merfish.obs['harmonized'].value_counts(dropna=False))

/opt/anaconda3/envs/neurospatial/lib/python3.11/site-packages/anndata/_core/anndata.py:1882: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Saved to: /Users/sydneycole/neuro/neuro/data/processed/main_merfish_obs_harmonized.csv

Harmonized distribution:
harmonized
None         7080435
EN-ET        2167483
IN           2021276
EN-IT-UL     1922584
EN-IT-DL     1704264
Astrocyte     517853
EC            368068
Glia          145407
Name: count, dtype: int64


In [2]:
# Fresh test load
test_load = load_main_merfish(PROJECT_ROOT)
print(f"Test load shape: {test_load.shape}")
print(f"Has harmonized column: {'harmonized' in test_load.obs.columns}")
print(test_load.obs['harmonized'].value_counts(dropna=False))

/opt/anaconda3/envs/neurospatial/lib/python3.11/site-packages/anndata/_core/anndata.py:1882: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/Users/sydneycole/neuro/neuro/src/data_loading.py:64: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  harmonized_df = pd.read_csv(harmonized_csv, index_col=0)


Test load shape: (15927370, 300)
Has harmonized column: True
harmonized
NaN          7080435
EN-ET        2167483
IN           2021276
EN-IT-UL     1922584
EN-IT-DL     1704264
Astrocyte     517853
EC            368068
Glia          145407
Name: count, dtype: int64
